In [35]:

import torch
from transformers import (
    GPT2LMHeadModel, 
    GPT2Tokenizer, 
    Trainer, 
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import Dataset
import pandas as pd
import numpy as np

In [36]:

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


In [37]:
def parse_qa_file(file_path):
    """
    Parse the Q&A file with context, questions, and answers.
    Returns a list of dictionaries with 'context', 'question', and 'answer' keys.
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # Split by story sections
    stories = content.split('Story: ')[1:]  # Skip empty first element

    dataset = []

    for story_block in stories:
        lines = story_block.strip().split('\n')

        # First line is the context/story
        context = lines[0].strip()

        # Parse Q&A pairs
        current_q = None
        for line in lines[1:]:
            line = line.strip()
            if line.startswith('Q: '):
                current_q = line[3:].strip()
            elif line.startswith('A: ') and current_q:
                answer = line[3:].strip()
                dataset.append({
                    'context': context,
                    'question': current_q,
                    'answer': answer
                })
                current_q = None

    return dataset

In [38]:
dataset_list = parse_qa_file('ETC-dataset-qa.txt')

print(f"Total Q&A pairs: {len(dataset_list)}")
print(f"\nFirst example:")
print(f"Context: {dataset_list[0]['context'][:100]}...")
print(f"Question: {dataset_list[0]['question']}")
print(f"Answer: {dataset_list[0]['answer'][:100]}...")

Total Q&A pairs: 927

First example:
Context: This section details the NICE clinical guidelines for the use of Electroconvulsive Therapy (ECT) in ...
Question: According to NICE, when should ECT be considered for depression?
Answer: ECT should be considered for the acute treatment of severe, life-threatening depression when a rapid...


In [39]:

def format_qa_for_clm(example):
    """
    Format the context, question, and answer into a single string for causal language modeling.
    Format: "Context: [context] \n Question: [question] \n Answer: [answer]"
    """
    formatted_text = (
        f"Context: {example['context']}\n"
        f"Question: {example['question']}\n"
        f"Answer: {example['answer']}"
    )
    return {'text': formatted_text}


In [40]:
formatted_data = [format_qa_for_clm(item) for item in dataset_list]

# Show formatted example
print("\nFormatted example:")
print(formatted_data[0]['text'][:300] + "...")

## Step 5: Create Hugging Face Dataset and Split

# Convert to Hugging Face Dataset
hf_dataset = Dataset.from_pandas(pd.DataFrame(formatted_data))


Formatted example:
Context: This section details the NICE clinical guidelines for the use of Electroconvulsive Therapy (ECT) in the treatment of depression, emphasizing the circumstances under which ECT should be considered, consent protocols, and clinical monitoring requirements.
Question: According to NICE, when sho...


In [41]:

# Split into train and validation sets (80/20 split)
train_test_split = hf_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

print(f"\nTrain size: {len(train_dataset)}")
print(f"Eval size: {len(eval_dataset)}")



Train size: 741
Eval size: 186


In [42]:

## Step 6: Load Tokenizer and Model

# Load GPT-2 tokenizer and model
model_name = "gpt2"  # You can also use "gpt2-medium", "gpt2-large", etc.
tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

# GPT-2 doesn't have a padding token by default, so we set it to the EOS token
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

print(f"\nModel loaded: {model_name}")
print(f"Vocab size: {len(tokenizer)}")
print(f"Pad token: {tokenizer.pad_token}")



Model loaded: gpt2
Vocab size: 50257
Pad token: <|endoftext|>


In [43]:

## Step 7: Tokenize the Dataset

def tokenize_function(examples):
    """
    Tokenize the text data.
    GPT-2 has a max length of 1024 tokens.
    """
    # Tokenize with truncation and padding
    result = tokenizer(
        examples['text'],
        truncation=True,
        max_length=512,  # Adjust based on your data and memory constraints
        padding=False,   # We'll use data collator for dynamic padding
    )

    # For causal language modeling, labels are the same as input_ids
    # result['labels'] = result['input_ids'].copy()
    # ***********************************************************************************************************
    # ***********************************************************************************************************
    # ***********************************************************************************************************
    # result['labels'] = list(result["input_ids"])

    return result

In [44]:

# Tokenize datasets
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_eval = eval_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=eval_dataset.column_names
)

print("\nDataset tokenized successfully!")


Map: 100%|██████████| 186/186 [00:00<00:00, 1902.89 examples/s]


Dataset tokenized successfully!


In [45]:

## Step 8: Create Data Collator

# Data collator will dynamically pad the inputs to the same length in each batch
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # We're doing causal language modeling, not masked language modeling
)


In [52]:

## Step 9: Set Training Arguments

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned-qa",           # Output directory
    overwrite_output_dir=True,
    num_train_epochs=3,                         # Number of training epochs
    per_device_train_batch_size=1,              # Batch size for training
    per_device_eval_batch_size=1,               # Batch size for evaluation
    warmup_steps=100,                           # Warmup steps
    learning_rate=5e-5,                         # Learning rate
    weight_decay=0.01,                          # Weight decay
    logging_dir='./logs',                       # Logging directory
    logging_steps=50,                           # Log every 50 steps
    evaluation_strategy="steps",                # Evaluate every eval_steps
    eval_steps=100,                             # Evaluate every 100 steps
    save_steps=200,                             # Save checkpoint every 200 steps
    save_total_limit=2,                         # Keep only the last 2 checkpoints
    load_best_model_at_end=True,                # Load best model at the end
    metric_for_best_model="eval_loss",          # Metric to determine best model
    greater_is_better=False,                    # Lower loss is better
    fp16=torch.cuda.is_available(),             # Use mixed precision if GPU available
    report_to="none",                           # Disable wandb/tensorboard
)

print("Training arguments configured!")

Training arguments configured!


In [53]:


## Step 10: Create Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
)

print("Trainer initialized!")


Trainer initialized!


In [54]:

## Step 11: Train the Model

print("\nStarting training...")
trainer.train()

print("\nTraining complete!")


  0%|          | 2/558 [03:17<15:14:58, 98.74s/it]



Starting training...


                                                 
  2%|▏         | 50/2223 [00:23<20:14,  1.79it/s]

{'loss': 3.475, 'learning_rate': 2.3000000000000003e-05, 'epoch': 0.07}


                                                  
  4%|▍         | 100/2223 [00:50<21:01,  1.68it/s]

...

{'loss': 0.7959, 'learning_rate': 6.59444182760245e-07, 'epoch': 2.97}

















































                                                   
                                             

 99%|█████████▉| 2200/2223 [16:29<00:05,  4.54it/s]



{'eval_loss': 1.1955702304840088, 'eval_runtime': 5.4344, 'eval_samples_per_second': 34.227, 'eval_steps_per_second': 34.227, 'epoch': 2.97}


                                                   
100%|██████████| 2223/2223 [16:38<00:00,  2.23it/s]

{'train_runtime': 998.2719, 'train_samples_per_second': 2.227, 'train_steps_per_second': 2.227, 'train_loss': 1.2734726448076241, 'epoch': 3.0}

Training complete!


In [55]:

## Step 12: Save the Fine-Tuned Model

# Save the model and tokenizer
model.save_pretrained("./gpt2-finetuned-qa-final")
tokenizer.save_pretrained("./gpt2-finetuned-qa-final")

print("\nModel and tokenizer saved to './gpt2-finetuned-qa-final'")


Model and tokenizer saved to './gpt2-finetuned-qa-final'


In [56]:


## Step 13: Inference with Fine-Tuned Model

def generate_answer(context, question, model, tokenizer, max_length=200):
    """
    Generate an answer given a context and question using the fine-tuned model.

    Args:
        context: The background/story context
        question: The question to answer
        model: The fine-tuned GPT-2 model
        tokenizer: The tokenizer
        max_length: Maximum length of generated text

    Returns:
        Generated answer text
    """
    # Format the input in the same way as training data
    input_text = f"Context: {context}\nQuestion: {question}\nAnswer:"

    # Tokenize the input
    input_ids = tokenizer.encode(input_text, return_tensors="pt").to(device)

    # Generate with the model
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=max_length,
            num_return_sequences=1,
            temperature=0.7,          # Controls randomness (lower = more deterministic)
            top_p=0.9,                # Nucleus sampling
            top_k=50,                 # Top-k sampling
            do_sample=True,           # Enable sampling
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3,   # Prevent repetition
        )

    # Decode the generated text
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract just the answer part (everything after "Answer:")
    if "Answer:" in generated_text:
        answer = generated_text.split("Answer:")[1].strip()
        # Stop at the end of the answer (if there's a newline or end token)
        answer = answer.split("\n")[0].strip()
        return answer

    return generated_text


In [57]:


# Load the fine-tuned model for inference
model = GPT2LMHeadModel.from_pretrained("./gpt2-finetuned-qa-final").to(device)
tokenizer = GPT2Tokenizer.from_pretrained("./gpt2-finetuned-qa-final")

model.eval()  # Set to evaluation mode

print("\nModel loaded for inference!")


Model loaded for inference!


In [58]:


## Step 14: Example Inference

# Test with a new example
test_context = """
This section details the NICE clinical guidelines for the use of Electroconvulsive Therapy (ECT) 
in the treatment of depression, emphasizing the circumstances under which ECT should be considered, 
consent protocols, and clinical monitoring requirements.
"""

test_question = "When should ECT be considered according to NICE guidelines?"

print("\n" + "="*80)
print("INFERENCE EXAMPLE")
print("="*80)
print(f"\nContext: {test_context.strip()}")
print(f"\nQuestion: {test_question}")
print(f"\nGenerated Answer:")

answer = generate_answer(test_context, test_question, model, tokenizer)
print(answer)


INFERENCE EXAMPLE

Context: This section details the NICE clinical guidelines for the use of Electroconvulsive Therapy (ECT) 
in the treatment of depression, emphasizing the circumstances under which ECT should be considered, 
consent protocols, and clinical monitoring requirements.

Question: When should ECT be considered according to NICE guidelines?

Generated Answer:


c:\Users\Anas Fareedi\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\generation\utils.py:1417: UserWarning: You have modified the pretrained model configuration to control generation. This is a deprecated strategy to control generation and will be removed soon, in a future version. Please use a generation configuration file (see https://huggingface.co/docs/transformers/main_classes/text_generation )
  warnings.warn(


ECT must be considered in severe, life-threatening or refractory depression when evidence suggests it may be better suited for life-saving or therapeutic interventions.


In [59]:


# Try another example
# test_question_2 = "What are the key consent principles for ECT?"
test_question_2 = "According to NICE, when should ECT be considered for depression?"

print("\n" + "-"*80)
print(f"\nQuestion: {test_question_2}")
print(f"\nGenerated Answer:")

answer_2 = generate_answer(test_context, test_question_2, model, tokenizer)
print(answer_2)

print("\n" + "="*80)
print("Inference complete!")
print("="*80)



--------------------------------------------------------------------------------

Question: According to NICE, when should ECT be considered for depression?

Generated Answer:
ECT may be considered in exceptional cases when the patient is seriously and life-threatening.

Inference complete!


In [ ]:
# ECT should be considered for the acute treatment of severe
# , life-threatening depression when a rapid response is required or when other treatment modalities have failed.